In [ ]:
# Cell 1 — Widget definitions with defaults
# Databricks Job base_parameters override these at runtime
# Defaults below are suitable for local run-ipynb testing (DBFS output)
dbutils.widgets.text("table_name", "default.test_export", "Table Name (catalog.schema.table)")
dbutils.widgets.text("output_path", "dbfs:/tmp/table-exporter-test/output", "Output Path")
dbutils.widgets.dropdown("file_format", "parquet", ["json", "csv", "parquet"], "Output Format")
dbutils.widgets.text("where_clause", "", "WHERE Clause (optional filter)")

In [ ]:
# Cell 2 — Read widget values
table_name = dbutils.widgets.get("table_name")
output_path = dbutils.widgets.get("output_path")
file_format = dbutils.widgets.get("file_format")
where_clause = dbutils.widgets.get("where_clause")

In [ ]:
# Cell 3 — Execute
# Ensure the test table exists (idempotent setup for local run-ipynb testing)
spark.sql("CREATE DATABASE IF NOT EXISTS default")
spark.sql("DROP TABLE IF EXISTS default.test_export")
spark.sql(
    "CREATE TABLE default.test_export USING DELTA "
    "AS SELECT id, name FROM VALUES (1, 'Alice'), (2, 'Bob') t(id, name)"
)

df = spark.table(table_name)
if where_clause:
    df = df.where(where_clause)
df.write.format(file_format).mode("overwrite").save(output_path)
print(f"Exported {table_name} to {output_path} as {file_format}")